<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/MaskPeoplePart1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mask people in the farm videos (part 1 of 4 (four people can run parts 1-4 at the same time))

Makes a copy of every video on the group drive with **every person blacked out**, so students and AI tools never see the faces or bodies of farm staff. Use the copies in `MaskedVideos/`, not the originals.

- Every frame written is checked for people (YOLO26s, "person"); each person gets a black box, a bit larger than them and held 0.5 s before and after.
- The copies are normal MP4s at 5 frames per second (the cameras change the picture only ~2 times a second). Every frame of the copy is checked, since an unchecked frame could show a person; only the check sheet is saved as pictures.
- Done videos are skipped; a stopped run continues where it stopped (every 30 min of video is saved).
- About 1-2 h per 4-hour video on a T4 GPU (an estimate: not timed yet).

Run the cells in order on a **T4 GPU**. No token is needed. Code: `CalvingPen/CalvingPipeline/Privacy.py`.

**Check the result** (cell 5): `MaskedVideos/Logs/<folder>/<video>/MaskCheck.jpg` shows masked frames at each new person and every 15 min. If a person is visible anywhere, do not share that video; tell the group.


In [ ]:
# 1. Setup: mount Drive, get the code from GitHub, install the libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"
if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from CalvingPipeline.Paths import FindVideo, MaskedVideoRoot
from CalvingPipeline.Privacy import ListAllVideos, MaskAll, MaskedPaths, SplitParts
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")


In [ ]:
# 2. Settings
OnlyVideo = None   # e.g. "CalvingPen3Sep26Cow1.avi" to test one video; None = all videos (or those of Part)
TryMinutes = None        # with OnlyVideo: only its first 30 min (listed as "trial"); None = the whole video
Part = 1            # None = all videos; 1-4 = only that quarter (same split for everyone)
Parts = 4
Settings = dict(
    OutFps=5,                # frames per second of the masked copy
    ModelName="yolo26s.pt",  # person detector (COCO class 0)
    ImageSize=1024,          # larger finds small / far-away people
    Confidence=0.15,         # low on purpose: a false alarm only blacks out a bit of pen
    HoldSeconds=0.5,         # the box stays 0.5 s before and after each sighting
    Style="Fill",            # "Fill" = black box (safest); "Pixelate" = coarse blocks
    SegmentMinutes=30,       # saved every 30 min of video (a stopped run continues from there)
    OutWidth=None,           # e.g. 1280 for smaller files; None = original size
)


In [ ]:
# 3. The videos on the drive and what is done. Status: to do / started / trial / done / empty (a recording with no picture).
Videos = ListAllVideos()
if OnlyVideo:
    Mine = {str(FindVideo(OnlyVideo))}
else:
    Mine = set(SplitParts(Videos, Parts)[Part - 1]) if Part else set(Videos.Source)
Videos["ThisRun"] = Videos.Source.isin(Mine)
print(f"{len(Videos)} videos on the drive, {Videos.Status.eq('done').sum()} done; this run: {len(Mine)} "
      f"({Videos[Videos.ThisRun].SizeGb.sum():.0f} GB)")
display(Videos[["Video", "Folder", "SizeGb", "Status", "ThisRun"]])


In [ ]:
# 4. Mask. Rerun after a disconnect: it continues where it stopped.
if OnlyVideo:
    Results = MaskAll(Videos=sorted(Mine), EndSeconds=TryMinutes * 60 if TryMinutes else None, **Settings)
else:
    Results = MaskAll(Part=Part, Parts=Parts, **Settings)
display(Results)


In [ ]:
# 5. Check: masked frames at each new person and every 15 min. Nobody should be visible.
for Source in sorted(Mine):
    Sheet = MaskedPaths(Source)[1] / "MaskCheck.jpg"
    if Sheet.exists():
        print(Path(Source).name)
        display(Image(str(Sheet), width=1200))
